In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_parquet('/content/drive/MyDrive/opinion-mining/train_cleaned.parquet')
df.head()

,sentence_id,split,domain,input,output,instruction
0,2339,train,laptops,I charge it at night and skip taking the cord ...,"{""aspects"": [{""term"": ""cord"", ""polarity"": ""neu...",Extract the aspect terms and their polarity fr...
1,812,train,laptops,I bought a HP Pavilion DV41222nr laptop and ha...,"{""aspects"": []}",Extract the aspect terms and their polarity fr...
2,1316,train,laptops,The tech guy then said the service center does...,"{""aspects"": [{""term"": ""service center"", ""polar...",Extract the aspect terms and their polarity fr...
3,2328,train,laptops,I investigated netbooks and saw the Toshiba NB...,"{""aspects"": []}",Extract the aspect terms and their polarity fr...
4,2193,train,laptops,The other day I had a presentation to do for a...,"{""aspects"": []}",Extract the aspect terms and their polarity fr...


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def run_qwen(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=65,
            do_sample=False,
            temperature = 0.0,
            top_k = None,
            top_p = None
        )

    result = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return result.strip()

In [ ]:
print(run_qwen("Classify sentiment: I love this movie"))

The sentiment of the statement "I love this movie" is positive.


In [ ]:
df = df.iloc[:2500]

In [ ]:
from tqdm import tqdm

results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Qwen inference"):
    text = row['input']

    prompt = "\n".join([
        "You are an information extraction system.",
        "",
        "Task: Extract the domain, ALL aspect terms, and their sentiment polarity from the text.",
        "",
        "IMPORTANT:",
        "- A text may contain MULTIPLE aspects",
        "- Extract EVERY aspect mentioned",
        "- Also predict the correct domain of the text",
        "",
        "Domains include (examples only):",
        "electronics, restaurants, movies, books, software, general",
        "",
        "Rules:",
        "- Output ONLY valid JSON",
        "- No explanation",
        "- Polarity: positive, negative, neutral",
        "- Domain must be one of: electronics, restaurants, movies, books, software, general",
        "",
        "Output format:",
        '{"domain":"...","aspects":[{"term":"...","polarity":"..."}, ...]}',
        "",
        "Example:",
        "Input: Boot time is fast but battery is bad.",
        'Output: {"domain":"electronics","aspects":[{"term":"Boot time","polarity":"positive"},{"term":"Battery","polarity":"negative"}]}',
        "",
        f"Text: {text}",
        "",
        "Output:"
    ])

    prediction = run_qwen(prompt)

    results.append({
        "input": text,
        "prediction": prediction
    })

Running Qwen inference: 100%|██████████| 2500/2500 [1:53:54<00:00,  2.73s/it]


In [ ]:
import json

with open("train_labeled.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download("train_labeled.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df2 = pd.read_json('/content/train_labeled.json')
df2.head()

,input,prediction
0,I charge it at night and skip taking the cord ...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ba..."
1,I bought a HP Pavilion DV41222nr laptop and ha...,"{""domain"":""electronics"",""aspects"":[{""term"":""HP..."
2,The tech guy then said the service center does...,"{""domain"":""electronics"",""aspects"":[{""term"":""se..."
3,I investigated netbooks and saw the Toshiba NB...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ne..."
4,The other day I had a presentation to do for a...,"{""domain"":""software"",""aspects"":[{""term"":""compu..."


In [ ]:
df2.iloc[0]['input']

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [ ]:
df2.iloc[0]['prediction']

'{"domain":"electronics","aspects":[{"term":"Battery life","polarity":"positive"}]}'